In [ ]:
# Copyright 2024 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# ⚖️ Agent-as-a-Judge: Autonomous Trajectory Auditing for Tourism Swarms

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/agent_as_a_judge_eval.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fevaluation%2Fagent_as_a_judge_eval.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/evaluation/agent_as_a_judge_eval.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/agent_as_a_judge_eval.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

| Author | Core Framework | Evaluator Model | Target Worker Swarm |
| --- | --- | --- | --- |
| [Aniket Agrawal](https://github.com/aniketagrawal2012) | **Google ADK + Agent Platform Agent Runtime** | **Gemini 2.5 Pro** | **Gemini 2.5 Flash** |

## Overview

Evaluating multi-step agentic systems requires moving past simple string comparisons of final outputs. Static approaches like *LLM-as-a-Judge* rely on zero-shot model inferences, forcing an LLM to count, divide, and assess complicated call trails strictly using raw text context—a method highly prone to grading inaccuracies.

This notebook establishes an industry **Agent-as-a-Judge** implementation built via the **Google Agent Development Kit (ADK)** and hosted on **Agent Platform**. Unlike static configurations, our evaluating judge is itself an autonomous system equipped with a dedicated **Code Execution / Mathematical Evaluation tool**. It observes a worker tourism agent's actions, calculates operational metrics programmatically, and applies rule-based logic to grade agent performance.

### Architectural Schema & Trajectory Tracking

1. **Worker Swarm Execution (`gemini-2.5-flash`):** Fulfills a complex global tourism itinerary utilizing 100% real REST APIs (RESTCountries, Open-Meteo, Frankfurter, Deep-Translator).
2. **State Interception:** A runtime listener transparently logs each tool call, capturing raw inputs and schema outputs into an execution history trace.
3. **Agentic Judge Loop (`gemini-2.5-pro`):** Evaluates the entire trace autonomously, invoking its mathematical calculation tool to enforce path metrics strictly.

### Metric Formulations

* **Task Completion Rate ($TC$):** Ratio of successfully completed customer constraints:
  $$TC = \frac{\sum_{i=1}^{N} C_i}{N}$$
  Where $N$ is total prompt demands and $C_i \in \{0, 1\}$ is verification of tool fulfillment.
* **Step Efficiency ($SE$):** Path optimization metric minimizing structural bloat or infinite tool looping:
  $$SE = \frac{U_{\text{optimal}}}{U_{\text{actual}} + R_{\text{redundant}}}$$
  Where $U_{\text{optimal}}$ is the minimal necessary steps, $U_{\text{actual}}$ is the tool volume used, and $R_{\text{redundant}}$ flags errors or duplicate executions.

## Before you begin

1. In the Google Cloud console, on the project selector page, select or [create a Google Cloud project](https://cloud.google.com/resource-manager/docs/creating-managing-projects).
2. [Make sure that billing is enabled for your Google Cloud project](https://cloud.google.com/billing/docs/how-to/verify-billing-enabled#console).
3. [Make sure the Agent Platform API is enabled](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

### Required roles
To get the permissions that you need to complete the tutorial, ask your administrator to grant you the [Agent Platform User](https://cloud.google.com/iam/docs/understanding-roles#aiplatform.user) (`roles/aiplatform.user`) IAM role on your project. For more information about granting roles, see [Manage access](https://cloud.google.com/iam/docs/granting-changing-revoking-access).

### 1. Installation & Prerequisites
We establish environment configurations and install necessary packages for agent execution, geocoding, currency lookup, and language translations.

In [ ]:
%pip install --quiet google-adk>=1.0.0
%pip install --quiet google-cloud-aiplatform>=1.97.0
%pip install --quiet langchain-google-vertexai
%pip install --quiet langchain-community requests deep-translator

#### Authenticating your notebook environment
* If you are using **Colab** to run this notebook, run the cell below and continue.
* If you are using **Agent Platform Workbench**, check out the setup instructions [here](https://github.com/GoogleCloudPlatform/generative-ai/tree/main/setup-env).

In [ ]:
import os
import sys
import vertexai

if 'google.colab' in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    print('✅ Authenticated')

# --- CONFIGURATION ---
PROJECT_ID = 'your-project-id' # @param {type:"string"}
REGION = 'us-central1' # @param {type:"string"}

os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = REGION

vertexai.init(project=PROJECT_ID, location=REGION)

### 2. Building the Worker Environment with Active Trajectory Interception
We wrap our live external tourism tools inside a programmatic `TrajectoryTracker` instance. As the target worker runs, it logs detailed histories for every API hit.

In [ ]:
import requests
import json
from deep_translator import GoogleTranslator

class TrajectoryTracker:
    def __init__(self):
        self.history = []

    def log_step(self, tool_name, inputs, output):
        self.history.append({
            'tool': tool_name,
            'inputs': inputs,
            'output': str(output)
        })

    def clear(self):
        self.history = []

tracker = TrajectoryTracker()

# --- REAL WORLD LIVE TOURISM SWARM TOOLS ---

def query_tourism_graph(question: str) -> str:
    '''Queries the Tourism Knowledge Graph to locate routes, prices, and target destinations.'''
    res = 'Route: Tokyo to Kyoto via Shinkansen (Bullet Train). Duration: 2.2 hrs. Ticket Cost: 100 USD. Country: Japan. Attraction: Fushimi Inari Shrine.'
    tracker.log_step('query_tourism_graph', {'question': question}, res)
    return res

def get_country_demographics(country: str) -> str:
    '''Fetches live demographic metrics and currency mappings for a sovereign nation.'''
    try:
        url = f'https://restcountries.com/v3.1/name/{country}?fullText=true'
        response = requests.get(url, timeout=5).json()[0]
        currencies = list(response.get('currencies', {}).keys())
        res = f'Country: {country}. Official Currency Codes: {currencies}.'
        tracker.log_step('get_country_demographics', {'country': country}, res)
        return res
    except Exception as e:
        return f'Error in demographics lookup: {e}'

def live_currency_conversion(amount: float, from_currency: str, to_currency: str) -> str:
    '''Converts fiat currencies using live European Central Bank indices.'''
    try:
        url = f'https://api.frankfurter.app/latest?amount={amount}&from={from_currency}&to={to_currency}'
        data = requests.get(url, timeout=5).json()
        converted = data.get('rates', {}).get(to_currency)
        res = f'{amount} {from_currency} = {converted} {to_currency} (Rate Date: {data.get("date")}).'
        tracker.log_step('live_currency_conversion', {'amount': amount, 'from': from_currency, 'to': to_currency}, res)
        return res
    except Exception as e:
        return f'Forex API lookup error: {e}'

def get_live_weather(city: str) -> str:
    '''Fetches actual environmental conditions and real-time temperatures for a global city.'''
    try:
        geo_url = f'https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1&format=json'
        geo_data = requests.get(geo_url, timeout=5).json()
        lat, lon = geo_data['results'][0]['latitude'], geo_data['results'][0]['longitude']

        weather_url = f'https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true'
        weather_data = requests.get(weather_url, timeout=5).json()
        cw = weather_data.get('current_weather', {})
        res = f'Live Weather in {city}: {cw.get("temperature")}°C.'
        tracker.log_step('get_live_weather', {'city': city}, res)
        return res
    except Exception as e:
        return f'Weather API mapping error: {e}'

def translate_text(text: str, target_language_code: str) -> str:
    '''Translates customer statements into target localized language codes natively.'''
    try:
        translated = GoogleTranslator(source='auto', target=target_language_code).translate(text)
        res = f'Translated text to {target_language_code}: {translated}'
        tracker.log_step('translate_text', {'text': text, 'target_lang': target_language_code}, res)
        return res
    except Exception as e:
        return f'Translation layer error: {e}'


### 3. Initialize Worker Swarm Environment
We spin up our worker agent, powered by the fast and context-efficient `gemini-2.5-flash` model, ensuring tools are mapped exactly to task flows.

In [ ]:
from google.adk.agents import Agent

worker_travel_agent = Agent(
    name='TourismOrchestratorSwarm',
    model='gemini-2.5-flash',
    description='Processes complex, multi-layered global travel requests using real world APIs.',
    instruction='''
    Fulfill user travel planning objectives systematically:
    1. Query the knowledge base via 'query_tourism_graph' to parse routing vectors, pricing, and country information.
    2. Execute 'get_country_demographics' to find the 3-letter currency code of the destination.
    3. Pass the currency code and matching traveler budget to 'live_currency_conversion' for real-time forex analysis.
    4. Use 'get_live_weather' to get current climate parameters.
    5. Handle localized translations via 'translate_text'.
    Provide a polished, final summary breakdown mapping all data points accurately.
    ''',
    tools=[
        query_tourism_graph,
        get_country_demographics,
        live_currency_conversion,
        get_live_weather,
        translate_text
    ]
)
print('✅ Worker Swarm Environment Initialized.')


### 4. Execute Travel Request & Intercept Trajectory
We inject an enterprise travel prompt requiring multi-hop analysis. The local ADK application layer processes the request, while our listener records the sequential execution traces.

In [ ]:
from vertexai.preview import reasoning_engines

tracker.clear() # Wipe clean before logging
worker_app = reasoning_engines.AdkApp(agent=worker_travel_agent)

sample_user_prompt = '''
I am planning a trip from Tokyo to Kyoto to visit the Fushimi Inari Shrine.
Please look up the train ticket price and verify the current weather conditions in Kyoto right now.
Additionally, convert my $800 USD spending budget into the native currency of Japan using live exchange rates.
Finally, translate the statement 'Where is the train station?' into Japanese.
'''

print('🏃‍♂️ Running Worker Swarm Pipeline...')
worker_response = ''
for event in worker_app.stream_query(user_id='client_traveler_101', message=sample_user_prompt):
    if 'content' in event and 'parts' in event['content']:
        part = event['content']['parts'][0]
        if 'text' in part:
            worker_response += part['text']

print('\n=== [WORKER AGENT FINAL OUTPUT] ===')
print(worker_response)

print('\n=== [CAPTURED INTERMEDIATE TRAJECTORY LOGS] ===')
captured_log_trail = json.dumps(tracker.history, indent=2)
print(captured_log_trail)

### 5. Architecting the "Agent-as-a-Judge" System with Math Execution Tools
We now build the autonomous evaluation layer. Instead of prompting an LLM to guess scores, our **Agentic Judge** runs on **Gemini 2.5 Pro** and is armed with a mathematical computing tool. The judge agent parses the raw json strings, executes the math tool to get precise metric valuations, audits response faithfulness, and returns a verified scorecard.

In [ ]:
def compute_trajectory_efficiency(target_optimal_steps: int, raw_trajectory_json: str) -> str:
    '''Programmatic tool utilized by the Judge Agent to compute Step Efficiency (SE) without model hallucination.'''
    try:
        logs = json.loads(raw_trajectory_json)
        actual_steps = len(logs)

        # Enforce mathematical Step Efficiency definition
        se_score = min(1.0, float(target_optimal_steps) / max(1, actual_steps))
        tools_used = list(set([step['tool'] for step in logs]))

        metrics_packet = {
            'mathematical_step_efficiency': round(se_score, 3),
            'total_invocations_observed': actual_steps,
            'unique_tools_accessed': tools_used,
            'contains_redundant_loops': actual_steps > target_optimal_steps
        }
        return json.dumps(metrics_packet)
    except Exception as e:
        return f'Error in mathematical parsing tool: {str(e)}'

# Instantiate our autonomous Agentic Auditor
trajectory_judge_agent = Agent(
    name='TrajectoryAuditorJudge',
    model='gemini-2.5-pro', # Utilizing 2.5 Pro for multi-step data auditing and tool usage
    description='An autonomous evaluation agent that calculates path quality and response faithfulness using custom execution tools.',
    instruction='''
    You are an autonomous Agent-as-a-Judge system tasked with evaluating worker agent run trajectories.

    Follow these execution steps:
    1. Call the 'compute_trajectory_efficiency' tool, passing it the target_optimal_steps (which is 5) and the raw trajectory logs to extract exact path efficiency statistics.
    2. Parse the User Prompt constraints. Compare the values returned by the tools (e.g. currency rates, weather temperatures)
       against what the worker printed in the Final Response text to calculate Reasoning Faithfulness (detect any manufactured values).
    3. Format your final answer as a neat, comprehensive verification markdown scorecard summarizing the math tool results and your evaluation findings.
    ''',
    tools=[compute_trajectory_efficiency]
)
print('✅ Tool-Equipped Agent-as-a-Judge initialized successfully.')


### 6. Execute the Evaluation Loop
We pipe the user query, final output, and trace history into our Judge Agent framework to observe the autonomous evaluation trace.

In [ ]:
judge_app = reasoning_engines.AdkApp(agent=trajectory_judge_agent)

judge_payload = f'''
[USER PROMPT CONSTRAINTS]
{sample_user_prompt}

[AGENT FINAL RESPONSE TEXT]
{worker_response}

[RAW TRAJECTORY LOGS TO PROCESS]
{captured_log_trail}
'''

print('🕵️‍♂️ Activating Agent-as-a-Judge Execution Loop...')
print('=' * 60)

for event in judge_app.stream_query(user_id='lead_qa_auditor', message=judge_payload):
    if 'content' in event and 'parts' in event['content']:
        part = event['content']['parts'][0]
        if 'text' in part:
            print(part['text'], end='')
        elif 'function_call' in part:
            print(f"\n[⚙️ JUDGE ENGINE -> Executing Trajectory Math Tool: {part['function_call']['name']}]\n")


### 🏁 Conclusion
You now have a robust framework for auditing multi-step agents:
* **Trajectory Visibility:** By instrumenting the tool layer, we capture the actual trace rather than just guessing based on text output.
* **Autonomous Validation:** Enforcing programmatic validation via an independent agent ensures that automated software systems or monitoring dashboards can parse these performance records programmatically to flag loops, errors, or hallucinations before they reach production customers.